# Teste de Instalação do `mustache-core` (TestPyPI)

Este notebook simula a experiência de um novo usuário instalando o pacote **MustaCHE Explorer** (`mustache-core`) a partir do TestPyPI e validando seu funcionamento básico em um ambiente isolado.

## 1. Instalação do Pacote

Para instalar o pacote e suas dependências necessárias a partir do PyPI padrão, utilize o comando abaixo. 

*Nota: Como o `core-sg` exige compilador C++ no Windows (caso não haja build pré-compilado para a sua versão do Python), a instalação pode falhar se você não tiver os C++ Build Tools instalados. Se isso ocorrer, você pode instalar o `mustache-core` ignorando as dependências de compilação ou usando um ambiente compatível.*

In [ ]:
# Instalação do TestPyPI com fallback de dependências no PyPI principal
%pip install --index-url https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple/ mustache-core==0.1.0

## 2. Importação e Verificação

Verificamos se o módulo `mustache` é importado corretamente e qual a versão instalada.

In [ ]:
import importlib.metadata
import sys

try:
    pkg_version = importlib.metadata.version('mustache-core')
    print(f"mustache-core versão instalada: {pkg_version}")
except importlib.metadata.PackageNotFoundError:
    print("mustache-core não está registrado nos metadados do pip.")

try:
    import mustache
    print("Módulo 'mustache' importado com sucesso!")
except ImportError as e:
    print(f"Falha ao importar o módulo 'mustache': {e}")

## 3. Verificação do Backend Core-SG

Verificamos se a biblioteca de aceleração de grafos `core-sg` está disponível.

In [ ]:
try:
    import core_sg
    print("Backend 'core-sg' está disponível e pronto para uso.")
except ImportError:
    print("Backend 'core-sg' NÃO está disponível (necessita de compiladores C++ para compilar módulos Cython no Windows).")

## 4. Teste de Execução de Agrupamento (Clustering)

Executamos um agrupamento simples com dados sintéticos bidimensionais gerados aleatoriamente.

In [ ]:
import pandas as pd
import numpy as np
from mustache.core.clustering import run_clustering

# Gerar dados sintéticos (dois grupos bem distintos)
np.random.seed(42)
blob1 = np.random.normal(loc=0.0, scale=0.5, size=(15, 2))
blob2 = np.random.normal(loc=5.0, scale=0.5, size=(15, 2))
dummy_data = np.vstack([blob1, blob2])
df = pd.DataFrame(dummy_data, columns=['x', 'y'])

# 4.1 Rodar algoritmo Padrão (HDBSCAN do scikit-learn)
print("Rodando clustering com backend padrão (scikit-learn)... ")
try:
    results_std = run_clustering(df, min_cluster_size=3, min_samples=3, algorithm='standard')
    print(f"Sucesso! Agrupamentos encontrados: {results_std['n_clusters']}, Ruídos: {results_std['noise_points']}")
except Exception as e:
    print(f"Erro no agrupamento padrão: {e}")

# 4.2 Rodar algoritmo com Core-SG (se disponível)
try:
    import core_sg
    print("\nRodando clustering com backend Core-SG... ")
    results_core = run_clustering(df, min_cluster_size=3, min_samples=3, algorithm='core-sg')
    print(f"Sucesso! Agrupamentos encontrados: {results_core['n_clusters']}, Ruídos: {results_core['noise_points']}")
except ImportError:
    print("\nIgnorando teste do Core-SG pois a dependência não está instalada.")
except Exception as e:
    print(f"Erro no agrupamento Core-SG: {e}")